# First-Five BioEmu Run: VS Code + Colab GPU, No Drive

The repository and dataset remain canonical on the Mac. VS Code uploads one temporary workspace ZIP to the Colab server, BioEmu runs on an **NVIDIA L4 or T4**, and one result ZIP is downloaded after each protein. Google Drive and a GitHub clone are not used.

Do not use **Run All** for the production section. Download each result archive before starting the next protein because `/content` is ephemeral. BioEmu conformations are candidate graph nodes; the shortest graph path is not a validated kinetic pathway.

## 1. Build And Upload The Workspace

Run this once in the normal local project terminal:

```bash
uv run python scripts/colab_transfer.py package-workspace
```

In VS Code, select a Colab server, complete Google authentication, then right-click the selected workspace archive under `outputs/colab_transfers/` and choose **Upload to Colab**. Set `EXPERIMENT_LABEL` in the next code cell to the matching archive stem. The workflow accepts `L4` or `T4`.

## 2. Verify The Runtime And Extract Source

In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import sys
import zipfile

CONTENT_ROOT = Path("/content")
# Use `non_ideal_transitions_n3` for the hard-transition batch.
EXPERIMENT_LABEL = "first_five_proteins"
UPLOAD_ARCHIVE = CONTENT_ROOT / f"{EXPERIMENT_LABEL}_workspace.zip"
WORKSPACE_DIR = CONTENT_ROOT / "protein-conformation-guided-random-walk"
ALLOWED_GPU_TYPES = ("L4", "T4")

if not CONTENT_ROOT.exists():
    raise RuntimeError("Select a Colab kernel before running this notebook.")
gpu_name = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
    text=True,
).strip()
normalized_gpu_name = gpu_name.upper()
matched_gpu_type = next((gpu for gpu in ALLOWED_GPU_TYPES if gpu in normalized_gpu_name), None)
print("Connected GPU:", gpu_name)
print("Allowed GPU types:", ", ".join(ALLOWED_GPU_TYPES))
if matched_gpu_type is None:
    raise RuntimeError(
        f"This workflow requires one of {ALLOWED_GPU_TYPES}; connected GPU is {gpu_name!r}."
    )
print("Matched GPU type:", matched_gpu_type)
if not UPLOAD_ARCHIVE.is_file():
    raise FileNotFoundError(
        f"Upload {EXPERIMENT_LABEL}_workspace.zip with the Colab extension: {UPLOAD_ARCHIVE}"
    )
if WORKSPACE_DIR.exists():
    shutil.rmtree(WORKSPACE_DIR)
with zipfile.ZipFile(UPLOAD_ARCHIVE) as handle:
    workspace_manifest = json.loads(handle.read("colab_workspace_manifest.json"))
    if workspace_manifest.get("schema_version") != "guided_conformation_colab_workspace_v1":
        raise RuntimeError("Unsupported workspace archive schema")
    handle.extractall(CONTENT_ROOT)
print("Extracted workspace:", WORKSPACE_DIR)
print("Dataset pairs:", workspace_manifest["pair_ids"] )

## 3. Install BioEmu And The Project

The first BioEmu call later downloads model and AlphaFold/ColabFold assets. Those caches remain available for the lifetime of this Colab server.

In [ ]:
# Colab preinstalls `tensorflow`; BioEmu installs `tensorflow-cpu` into the same
# package namespace. Remove both variants first to avoid mixed native binaries.
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "uninstall",
        "-y",
        "tensorflow",
        "tensorflow-cpu",
        "tf-keras",
    ],
    check=False,
)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-e",
        f"{WORKSPACE_DIR}[notebook]",
        "bioemu[cuda]",
    ],
    check=True,
)
required_adapter = WORKSPACE_DIR / "guided_conformational_path/dataset_bioemu_adapter.py"
if not required_adapter.is_file():
    raise RuntimeError(f"Workspace is missing the dataset adapter: {required_adapter}")
# Editable-install path files are processed only when Python starts. Make the
# freshly extracted package importable without restarting the Colab kernel.
if str(WORKSPACE_DIR) not in sys.path:
    sys.path.insert(0, str(WORKSPACE_DIR))
print("Installation complete on", gpu_name)

## 4. Define One-Pair Sampling, Graph, And Export

Every call writes to a sample-count-specific directory under `/content/runs`, then creates a self-contained ZIP under `/content/exports`. Existing completed bundles are reused within the current runtime. Keep one pair/sample-count task per cell, especially on `T4`, because BioEmu still cannot checkpoint mid-sample.

In [ ]:
from guided_conformational_path.bioemu_runner import run_bioemu_from_fasta
from guided_conformational_path.colab_transfer import create_result_archive, sha256_file
from guided_conformational_path.dataset_bioemu_experiment import (
    DatasetBioEmuConfig,
    run_dataset_bioemu_experiment,
)
from guided_conformational_path.large_synthetic import format_run_summary, write_run_outputs

DATASET_ROOT = WORKSPACE_DIR / "data/dataset" / EXPERIMENT_LABEL
RUNS_ROOT = CONTENT_ROOT / "runs"
EXPORTS_ROOT = CONTENT_ROOT / "exports"
GRAPH_K_NEIGHBORS = 10
WRITE_VISUAL_OUTPUTS = True
PAIR_MANIFESTS = {
    path.parent.name: path
    for path in sorted(DATASET_ROOT.glob("*/pairs/*/manifest.json"))
}
if not PAIR_MANIFESTS:
    raise RuntimeError(f"No pair manifests found under {DATASET_ROOT}")

def run_pair(pair_id: str, n_samples: int) -> dict[str, object]:
    pair_manifest_path = PAIR_MANIFESTS[pair_id]
    pair_manifest = json.loads(pair_manifest_path.read_text(encoding="utf-8"))
    pair_dir = pair_manifest_path.parent
    protein_id = pair_manifest["protein_id"]
    fasta_path = pair_dir / pair_manifest["paths"]["bioemu_sequence"]
    run_name = f"n_{n_samples}"
    bundle_dir = RUNS_ROOT / "bioemu_bundles" / protein_id / pair_id / run_name
    graph_dir = RUNS_ROOT / "graph_outputs" / protein_id / pair_id / run_name
    archive_path = EXPORTS_ROOT / f"{protein_id}__{pair_id}__{run_name}.zip"

    print(f"\n[{protein_id} / {pair_id}] requesting {n_samples} samples")
    if (bundle_dir / "manifest.json").exists():
        print("Reusing completed bundle:", bundle_dir)
    else:
        run_bioemu_from_fasta(
            fasta_path=fasta_path,
            output_dir=bundle_dir,
            n_samples=n_samples,
            seed=7,
        )

    config = DatasetBioEmuConfig(
        dataset_pair_dir=pair_dir,
        bioemu_bundle_dir=bundle_dir,
        max_bioemu_samples=n_samples,
        k_neighbors=GRAPH_K_NEIGHBORS,
        outputs_dir=graph_dir,
        write_nodes=False,
        write_pymol=WRITE_VISUAL_OUTPUTS,
        write_py3dmol=WRITE_VISUAL_OUTPUTS,
        write_animation=WRITE_VISUAL_OUTPUTS,
    )
    run = run_dataset_bioemu_experiment(config)
    write_run_outputs(run, config)
    create_result_archive(bundle_dir, graph_dir, archive_path, dataset_label=EXPERIMENT_LABEL)
    result = {
        "protein_id": protein_id,
        "pair_id": pair_id,
        "requested_samples": n_samples,
        "retained_samples": run.summary["n_generated_nodes"],
        "path": run.path_result.node_ids,
        "total_cost": run.path_result.total_cost,
        "archive": str(archive_path),
        "archive_sha256": sha256_file(archive_path),
    }
    print(format_run_summary(run, graph_dir))
    print(json.dumps(result, indent=2))
    print("DOWNLOAD THIS FILE BEFORE CONTINUING:", archive_path)
    return result

## 5. Required Smoke Test

Run this first. In the Colab Contents panel, open `/content/exports`, right-click the resulting ZIP, and choose **Download...**. Save it under local `outputs/colab_transfers/downloads/`.

In [ ]:
smoke_result = run_pair("4AKE_A__1AKE_A", n_samples=2)

Import the downloaded smoke archive from the local terminal before production:

```bash
uv run python scripts/colab_transfer.py import-result outputs/colab_transfers/downloads/P69441__4AKE_A__1AKE_A__n_2.zip
```

# Production: 200 Requested Samples Per Protein

Run one numbered cell at a time. Download and locally import its ZIP before running the next cell. BioEmu can retain fewer than 200 samples after filtering.

## 6. P69441, 214 residues

In [ ]:
p69441_result = run_pair("4AKE_A__1AKE_A", n_samples=200)

Download `/content/exports/P69441__4AKE_A__1AKE_A__n_200.zip`, import it locally, then continue.

## 7. P06746, 335 residues

In [ ]:
p06746_result = run_pair("1BPX_A__1BPY_A", n_samples=200)

Download `/content/exports/P06746__1BPX_A__1BPY_A__n_200.zip`, import it locally, then continue.

## 8. P0AEX9, 396 residues

In [ ]:
p0aex9_result = run_pair("1OMP_A__1ANF_A", n_samples=200)

Download `/content/exports/P0AEX9__1OMP_A__1ANF_A__n_200.zip`, import it locally, then continue.

## 9. P00889, 464 residues

In [ ]:
p00889_result = run_pair("1CTS_A__2CTS_A", n_samples=200)

Download `/content/exports/P00889__1CTS_A__2CTS_A__n_200.zip`, import it locally, then continue.

## 10. P35557, 465 residues

In [ ]:
p35557_result = run_pair("1V4T_A__1V4S_A", n_samples=200)

Download `/content/exports/P35557__1V4T_A__1V4S_A__n_200.zip` and import it locally. Production is complete when all five imported `experiment_summary.json` files exist under `outputs/dataset_bioemu/`.

## 11. Non-Ideal Transitions Batch

Set `EXPERIMENT_LABEL = "non_ideal_transitions_n3"` in section 4, rerun that section to reload its pair manifests, then run one T4 task at a time below. Download each archive before moving to the next task. KaiB is mutation-stabilized and both Mad2 endpoints are NMR; RfaH is a CTD-only comparison. These are QC caveats, not failures of the workflow.

In [ ]:
mad2_result = run_pair("1DUJ_A__1S2H_A", n_samples=200)

In [ ]:
kaib_result = run_pair("2QKE_A__5JYT_A", n_samples=200)

In [ ]:
rfah_ctd_result = run_pair("2OUG_A__2LCL_A", n_samples=200)

## Local Import Command Template

Run for each downloaded production ZIP:

```bash
uv run python scripts/colab_transfer.py import-result outputs/colab_transfers/downloads/<archive-name>.zip
```

Use `--force` only when intentionally replacing a previously imported run.

## Release The Legacy Runtime

Run the next cell immediately after confirming that every required ZIP has finished downloading. This legacy workflow cannot detect completion of a VS Code download from inside Colab, so releasing earlier could destroy an undownloaded archive. The Hugging Face worker notebook releases automatically after remote verification.

In [ ]:
from google.colab import runtime

print('Downloads confirmed; disconnecting and deleting this Colab runtime.', flush=True)
runtime.unassign()